In [1]:
!wget -q -O requirements-cpu.txt https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/homework/08-deep-learning/requirements-cpu.txt
!wget -q -O reference_train.py https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/homework/08-deep-learning/reference_train.py

In [2]:
!uv pip install --python .venv311/bin/python --index-strategy unsafe-best-match -r requirements-cpu.txt

Using Python 3.11.18 environment at: .venv311
Checked 4 packages in 118ms


In [3]:
!uv pip install --python .venv311/bin/python ipykernel
!.venv311/bin/python -m ipykernel install --user --name mlzoomcamp-py311 --display-name "Python 3.11 (ML Zoomcamp)"

Using Python 3.11.18 environment at: .venv311
Checked 1 package in 45ms
Installed kernelspec mlzoomcamp-py311 in /home/codespace/.local/share/jupyter/kernels/mlzoomcamp-py311


In [4]:
import sys
import torch
import torchvision
import numpy as np
import PIL

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)

Python: 3.11.18 (main, Nov 11 2026, 00:16:37) [Clang 23.1.3 ]
PyTorch: 2.9.0+cpu
Torchvision: 0.24.0+cpu
NumPy: 2.3.3
Pillow: 11.3.0


In [5]:
!wget -q -O data.zip https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip

In [6]:
import hashlib

checksum = hashlib.sha256(open("data.zip", "rb").read()).hexdigest()
print(checksum)

assert checksum == "9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67"

9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67


In [7]:
!unzip -q -o data.zip

In [8]:
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder("data/train")
evaluation_dataset = ImageFolder("data/test")

print("Class mapping:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Evaluation images:", len(evaluation_dataset))

assert train_dataset.class_to_idx == {"curly": 0, "straight": 1}
assert len(train_dataset) == 800
assert len(evaluation_dataset) == 201

Class mapping: {'curly': 0, 'straight': 1}
Training images: 800
Evaluation images: 201


In [9]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()
print(criterion)

BCEWithLogitsLoss()


In [10]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = HairModel()
parameter_count = sum(p.numel() for p in model.parameters())
print(parameter_count)

20073473


In [11]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "reference_train.py", "--data-dir", "data"],
    check=True
)

{
  "output_dir": "runs/reference",
  "parameters": 20073473,
  "baseline_median_train_accuracy": 0.808125,
  "baseline_train_loss_std_population": 0.13934077404079703,
  "augmented_mean_evaluation_loss": 0.5749407216832413,
  "augmented_last_five_evaluation_accuracy": 0.7194029850746269
}


CompletedProcess(args=['/workspaces/Ml-zoomcamp/.venv311/bin/python', 'reference_train.py', '--data-dir', 'data'], returncode=0)

In [12]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "reference_train.py", "--data-dir", "data"],
    check=True
)

{
  "output_dir": "runs/reference",
  "parameters": 20073473,
  "baseline_median_train_accuracy": 0.808125,
  "baseline_train_loss_std_population": 0.13934077404079703,
  "augmented_mean_evaluation_loss": 0.5749407216832413,
  "augmented_last_five_evaluation_accuracy": 0.7194029850746269
}


CompletedProcess(args=['/workspaces/Ml-zoomcamp/.venv311/bin/python', 'reference_train.py', '--data-dir', 'data'], returncode=0)

In [13]:
import json
import numpy as np

with open("runs/reference/history_baseline.json") as file:
    baseline = json.load(file)

with open("runs/reference/history_augmented.json") as file:
    augmented = json.load(file)

question_3 = round(float(np.median(baseline["train_accuracy"])), 2)
question_4 = round(float(np.std(baseline["train_loss"], ddof=0)), 3)
question_5 = round(float(np.mean(augmented["evaluation_loss"])), 3)
question_6 = round(float(np.mean(augmented["evaluation_accuracy"][-5:])), 2)

print("Question 1:", "nn.BCEWithLogitsLoss()")
print("Question 2:", parameter_count)
print("Question 3:", question_3)
print("Question 4:", question_4)
print("Question 5:", question_5)
print("Question 6:", question_6)

Question 1: nn.BCEWithLogitsLoss()
Question 2: 20073473
Question 3: 0.81
Question 4: 0.139
Question 5: 0.575
Question 6: 0.72


In [14]:
from pathlib import Path

print(Path("runs/reference/history_baseline.json").exists())
print(Path("runs/reference/history_augmented.json").exists())

True
True
